In [1]:
import pandas as pd
import requests
import zipfile
import io
import xlrd
import csv

/Users/celine/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# Run this to download mrt.csv
STATION_DATA_ENDPOINT = (
    "https://datamall.lta.gov.sg/content/dam/datamall/datasets/Geospatial/"
    "Train%20Station%20Codes%20and%20Chinese%20Names.zip"
)

def get_station_names(endpoint: str) -> list[tuple[str, str]]:
    """Download train station codes and station names.

    Args:
        endpoint (str): HTTPS address of zipped XLS file containing train station codes and names.

    Returns:
        list[tuple[str, str]]: Train stations sorted by station code in ascending order.
        For example, ("CC1", "Dhoby Ghaut"), ("NE6", "Dhoby Ghaut"), ("NS24", "Dhoby Ghaut").
    """
    with requests.Session() as session:
        res = session.get(endpoint, timeout=30)
        res.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(res.content), "r") as z:
        excel_bytes = z.read(
            z.infolist()[0]
        )  # Zip file should only contain one XLS file.
        workbook = xlrd.open_workbook(file_contents=excel_bytes)
        sheet = workbook.sheet_by_index(0)

    stations: set[tuple[str, str]] = {
        (sheet.cell_value(row_idx, 0).strip(), sheet.cell_value(row_idx, 1).strip())
        for row_idx in range(1, sheet.nrows)
    }

    return sorted(
        stations,
        key=lambda station: station[0],
    )


def get_coordinates_onemap(location_name):
    endpoint = "https://www.onemap.gov.sg/api/common/elastic/search"

    res = requests.get(
        endpoint,
        params={
            "searchVal": location_name,
            "returnGeom": "Y",
            "getAddrDetails": "Y",
            "pageNum": "1",
        },
        timeout=15,
    ).json()
    results = res.get("results", None)
    if isinstance(results, list):
        for result in results:
            if "LATITUDE" in result and "LONGITUDE" in result:
                return float(result["LATITUDE"]), float(result["LONGITUDE"])
    return None

stations = {
        station: {
            "lat": None,
            "lon": None,
        }
        for station in get_station_names(STATION_DATA_ENDPOINT)
    }

# Get all operational stations coordinates from OneMap.
for (station_code, station_name), station_details in stations.items():
    coordinates = None
    location_name = station_name + " " + station_code
    try:
        coordinates = get_coordinates_onemap(location_name)
        if coordinates:
            stations[(station_code, station_name)]["lat"] = coordinates[0]
            stations[(station_code, station_name)]["lon"] = coordinates[1]
    except Exception as e:
        _ = e
## Run this to download mrt.csv
with open("../../data/raw/mrt.csv", "w") as f:
    csv_writer = csv.writer(f)
    csv_writer.writerow(
        ("station_code", "station_name", "lat", "lon")
    )
    csv_writer.writerows(
        sorted(
            (
                (
                    station_code,
                    station_name,
                    *details.values(),
                )
                for (
                    station_code,
                    station_name,
                ), details in stations.items()
            ),
            key=lambda x: x[0])
        )
mrt = pd.read_csv("../../data/raw/mrt.csv", index_col = 0).reset_index()
mrt.head()

,station_code,station_name,lat,lon
0,BP1,Choa Chu Kang,1.384755,103.744538
1,BP10,Fajar,1.384573,103.770887
2,BP11,Segar,1.387785,103.769600
3,BP12,Jelapang,1.386739,103.764534
4,BP13,Senja,1.382725,103.762344


In [3]:
# check for missing lat lon
mrt[mrt["lat"].isna() | mrt["lon"].isna()]

,station_code,station_name,lat,lon


In [4]:
# save to csv
mrt.to_csv("../../data/cleaned/mrt_cleaned.csv", index=False)